# Notebook 2 - Product Price Analysis

**Project:** Web Scraping and Product Price Analysis System

**Run order:** `scraping/web_scraping.ipynb` (1) -> **this notebook (2)** -> `dashboard/Product_Price_Analysis_Dashboard.ipynb` (3)

**Input:** `data/scraped_product_data.csv` (created by Notebook 1)  
**Outputs:** 4 CSV files in `results/` and 5 charts in `visualizations/`

> **Status note:** this notebook was written from the project description. The `pd.cut()` price-range
> boundaries (Section 8) are an assumption - equal-width bins - so replace them with your own if you used fixed values.

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

def find_project_root():
    """Return the repository root whether the notebook runs from a sub-folder
    (local / cloned repo) or from /content (a fresh Google Colab session)."""
    cwd = Path.cwd()
    return cwd.parent if cwd.name in {"scraping", "analysis", "dashboard"} else cwd

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "scraped_product_data.csv"
XLSX_PATH = DATA_DIR / "scraped_product_data.xlsx"
print("Project root :", PROJECT_ROOT)
print("Data folder  :", DATA_DIR)

RESULTS_DIR = PROJECT_ROOT / "results"
VIZ_DIR = PROJECT_ROOT / "visualizations"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

## 2. Load the dataset

If the CSV is not found (for example, in a fresh Colab session), you can upload it.

In [ ]:
if not CSV_PATH.exists():
    try:
        from google.colab import files
        print("scraped_product_data.csv not found - please upload it now.")
        uploaded = files.upload()
        for name, content in uploaded.items():
            if name.endswith(".csv"):
                CSV_PATH.write_bytes(content)
    except ImportError:
        raise FileNotFoundError(
            f"{CSV_PATH} not found. Run scraping/web_scraping.ipynb first.")

df = pd.read_csv(CSV_PATH)
print("Loaded:", CSV_PATH)
print("Shape :", df.shape)
df.head()

## 3. Clean the Price (and Rating) columns

Invalid values become `NaN` and are excluded from calculations, rather than crashing the notebook.

In [ ]:
def to_number(series):
    cleaned = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

required = ["Product_Name", "Vendor", "Category", "Price", "Rating"]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"Missing columns in CSV: {missing_cols}"

df["Price"] = to_number(df["Price"])
df["Rating"] = to_number(df["Rating"])

invalid_prices = df["Price"].isna().sum()
print("Rows with invalid/missing Price :", invalid_prices)
print("Rows with invalid/missing Rating:", df["Rating"].isna().sum())

# Price-based analysis only uses rows that have a valid price.
df = df.dropna(subset=["Price"]).reset_index(drop=True)
print("Rows used for price analysis    :", len(df))

## 4. Basic price statistics

In [ ]:
average_price = df["Price"].mean()
minimum_price = df["Price"].min()
maximum_price = df["Price"].max()
median_price = df["Price"].median()

print(f"Average price: {average_price:.2f}")
print(f"Minimum price: {minimum_price:.2f}")
print(f"Maximum price: {maximum_price:.2f}")
print(f"Median price : {median_price:.2f}")

df["Price"].describe().round(2)

## 5. Cheapest and most expensive product

In [ ]:
show_cols = ["ID", "Product_Name", "Vendor", "Category", "Price", "Rating"]
show_cols = [c for c in show_cols if c in df.columns]

cheapest = df.loc[df["Price"].idxmin(), show_cols]
most_expensive = df.loc[df["Price"].idxmax(), show_cols]

print("Cheapest product:\n", cheapest.to_string(), "\n")
print("Most expensive product:\n", most_expensive.to_string())

## 6. Product price distribution (histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["Price"], bins=20, edgecolor="black")
ax.set_title("Product Price Distribution")
ax.set_xlabel("Price")
ax.set_ylabel("Number of Products")
fig.tight_layout()
fig.savefig(VIZ_DIR / "price_distribution.png", dpi=150)
plt.show()

## 7. Price by category

In [ ]:
avg_price_by_category = (df.groupby("Category")["Price"].mean()
                           .sort_values(ascending=False))
print(avg_price_by_category.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
avg_price_by_category.plot(kind="bar", ax=ax, edgecolor="black")
ax.set_title("Average Price by Category")
ax.set_xlabel("Category")
ax.set_ylabel("Average Price")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(VIZ_DIR / "average_price_by_category.png", dpi=150)
plt.show()

In [ ]:
# Minimum, maximum and average price per category.
category_price_analysis = (df.groupby("Category")["Price"]
                             .agg(Minimum_Price="min", Maximum_Price="max", Average_Price="mean")
                             .round(2)
                             .reset_index())
category_price_analysis

## 8. Top 10 expensive and cheapest products

In [ ]:
top_10_expensive = df.nlargest(10, "Price")[show_cols].reset_index(drop=True)
top_10_cheapest = df.nsmallest(10, "Price")[show_cols].reset_index(drop=True)

print("Top 10 most expensive products")
display(top_10_expensive)
print("Top 10 cheapest products")
display(top_10_cheapest)

## 9. Price ranges (Low / Medium / High)

`pd.cut(bins=3)` splits the price range into 3 **equal-width** groups.  
*Assumption:* if your original code used fixed boundaries (for example `[0, 50, 150, max]`), replace `bins=3` with them.

In [ ]:
labels = ["Low", "Medium", "High"]
df["Price_Range"] = pd.cut(df["Price"], bins=3, labels=labels)

products_by_price_range = df["Price_Range"].value_counts().reindex(labels)
print(products_by_price_range)

fig, ax = plt.subplots(figsize=(7, 5))
products_by_price_range.plot(kind="bar", ax=ax, edgecolor="black")
ax.set_title("Products by Price Range")
ax.set_xlabel("Price Range")
ax.set_ylabel("Number of Products")
plt.xticks(rotation=0)
fig.tight_layout()
fig.savefig(VIZ_DIR / "products_by_price_range.png", dpi=150)
plt.show()

## 10. Price vs rating

In [ ]:
rating_data = df.dropna(subset=["Rating"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(rating_data["Price"], rating_data["Rating"], alpha=0.6)
ax.set_title("Price vs Rating")
ax.set_xlabel("Price")
ax.set_ylabel("Rating")
fig.tight_layout()
fig.savefig(VIZ_DIR / "price_vs_rating.png", dpi=150)
plt.show()

price_rating_correlation = rating_data["Price"].corr(rating_data["Rating"])
print("Price-Rating correlation:", round(price_rating_correlation, 4))

## 11. Price by vendor

In [ ]:
avg_price_by_vendor = (df.groupby("Vendor")["Price"].mean()
                         .sort_values(ascending=False))
print(avg_price_by_vendor.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
avg_price_by_vendor.plot(kind="bar", ax=ax, edgecolor="black")
ax.set_title("Average Price by Vendor")
ax.set_xlabel("Vendor")
ax.set_ylabel("Average Price")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(VIZ_DIR / "average_price_by_vendor.png", dpi=150)
plt.show()

## 12. Final summary table

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total Products Analysed", "Average Price", "Minimum Price", "Maximum Price",
        "Median Price", "Cheapest Product", "Most Expensive Product",
        "Price-Rating Correlation", "Number of Categories", "Number of Vendors",
    ],
    "Value": [
        len(df), round(average_price, 2), round(minimum_price, 2), round(maximum_price, 2),
        round(median_price, 2), cheapest["Product_Name"], most_expensive["Product_Name"],
        round(price_rating_correlation, 4), df["Category"].nunique(), df["Vendor"].nunique(),
    ],
})
summary

## 13. Export results to CSV

In [ ]:
summary.to_csv(RESULTS_DIR / "product_price_analysis_summary.csv", index=False)
category_price_analysis.to_csv(RESULTS_DIR / "category_price_analysis.csv", index=False)
top_10_expensive.to_csv(RESULTS_DIR / "top_10_expensive_products.csv", index=False)
top_10_cheapest.to_csv(RESULTS_DIR / "top_10_cheapest_products.csv", index=False)

print("Results saved in:", RESULTS_DIR)
print("Charts saved in :", VIZ_DIR)
for p in sorted(RESULTS_DIR.glob("*.csv")) + sorted(VIZ_DIR.glob("*.png")):
    print(" -", p.name)

In [ ]:
# Colab only: download everything, because Colab files disappear when the session ends.
try:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/analysis_outputs", "zip", PROJECT_ROOT, "results")
    shutil.make_archive("/content/analysis_charts", "zip", PROJECT_ROOT, "visualizations")
    files.download("/content/analysis_outputs.zip")
    files.download("/content/analysis_charts.zip")
except ImportError:
    print("Not running in Colab - files are already in results/ and visualizations/.")

**Next step:** open `dashboard/Product_Price_Analysis_Dashboard.ipynb`. It uses the same CSV.